[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/notebooks/RO/capitol2_notebook_seminar.ipynb)

# Modelarea piețelor financiare — Seminarul 2: Eficiența pieței

*Notebook de seminar. Daniel Traian Pele, Antoaneta Amza, Academia de Studii Economice din București.*

- Problemele **[Rezolvat]** conțin rezolvarea completă; problemele **[Propus]** le rezolvați individual.
- Partea A: pe hîrtie și derivări (A1–A6); Partea B: calcul și inferență (B1–B9); Partea C: întrebări deschise și critica unui răspuns artificial intelligence (AI) (C2, C3; verificarea C1 folosește codul din B1).

Exercițiile sînt grupate după exemplul rezolvat care le pregătește; codurile originale rămîn pentru legăturile cu slide-urile. Unele exerciții propuse cer replicarea unui reper public din curs: aveți de prezentat implementarea și interpretarea proprii.

> **Lucrați în Google Colab? Salvați mai întîi copia proprie.**
> Linkul Colab deschide o copie temporară: modificările se pierd cînd închideți fila sau expiră sesiunea.
> Folosiți **File → Save a copy in Drive** (copia ajunge în folderul *Colab Notebooks* din Google Drive-ul ASE) și lucrați în acea copie.
> Pentru proiect, **File → Save a copy in GitHub** salvează notebookul direct în repository-ul echipei.
> Fișierele create în timpul execuției (CSV, grafice) sînt tot temporare: setați `SAVE_TO_DRIVE = True` în celula următoare pentru a le păstra.

In [ ]:
# Opțional (doar în Colab): păstrați în Google Drive fișierele create
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/MFM/Chapter_2'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


Save your outputs to: /content/mfm-audit/20261016T142212-a769b9


> Rezolvările se discută la seminar.

In [ ]:
import io
import os
import re
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats
from scipy.special import gammaln
import warnings
warnings.filterwarnings('ignore')

## Stil și funcții ajutătoare

In [ ]:
# Stilul graficelor: fundal transparent, legenda sub grafic
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial', 'Helvetica']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.1
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 8

# Chart colours
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Crimson  = '#DC3545'
Navy     = '#1F2A44'   # reference lines (no grey in charts)
BandBlue = '#C5D2E8'   # light MainBlue tint for confidence / reference bands
Gray, LightGray = Navy, BandBlue   # legacy names kept for importing scripts
GROUP_COL = {'Developed': MainBlue, 'Emerging/frontier': IDAred, 'Crypto': Amber, 'FX': Forest}

SEED = 42
TABLE_DIR = '.'


def save_fig(name):
    """Save the figure as transparent PDF and PNG in the current folder."""
    plt.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()
    print(f"   saved {name}")


def legend_outside_bottom(ax, ncol=2, y=-0.22):
    """Place the legend outside the plot, bottom centre."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)



import tempfile
KEEP = globals().get('SAVE_TO_DRIVE', False)   # comutatorul din celula Colab de mai sus
TABLE_DIR = globals().get('OUT_DIR', '.') if KEEP else tempfile.gettempdir()


def save_fig(name):
    """Afiseaza figura; cu SAVE_TO_DRIVE = True o salveaza si (PDF si PNG) in OUT_DIR."""
    if KEEP:
        for ext in ('pdf', 'png'):
            plt.savefig(os.path.join(TABLE_DIR, f'{name}.{ext}'), bbox_inches='tight', transparent=True, dpi=200)
    plt.show()
    plt.close()

## Date

- Închideri zilnice pînă la 18 septembrie 2026.
- Indici bursieri: doar zilele lucrătoare; o închidere egală cu cea anterioară este tratată ca zi nelucrătoare în care s-a repetat prețul anterior și eliminată (se elimină astfel și rarele zile reale cu randament zero).
- Cripto se tranzacționează 7 zile din 7; EUR/RON este cursul oficial de referință al Băncii Naționale a României (BNR).
- Fiecare serie este analizată pe propriul calendar.

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
# local copy of the course data, if the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'

# name -> (symbol, label, group, start date, days per year)
MARKETS = {
    'sp500':  ('GSPC.INDX',     'S&P 500 (US)',          'Developed', '2000-01-01', 252),
    'stoxx':  ('STOXX50E.INDX', 'Euro Stoxx 50',         'Developed', '2000-01-01', 252),
    'nikkei': ('N225.INDX',     'Nikkei 225 (Japan)',    'Developed', '2000-01-01', 252),
    'hsi':    ('HSI.INDX',      'Hang Seng (HK)',        'Developed', '2000-01-01', 252),
    'bet':    ('BET',           'BET (Romania)',         'Emerging/frontier', '2000-01-01', 252),
    'wig20':  ('WIG20.INDX',    'WIG20 (Poland)',        'Emerging/frontier', '2000-01-01', 252),
    'bux':    ('BUX.INDX',      'BUX (Hungary)',         'Emerging/frontier', '2000-01-01', 252),
    'px':     ('PX.INDX',       'PX (Czechia)',          'Emerging/frontier', '2000-01-01', 252),
    'xu100':  ('XU100.INDX',    'BIST 100 (Turkey)',     'Emerging/frontier', '2000-01-01', 252),
    'bvsp':   ('BVSP.INDX',     'Bovespa (Brazil)',      'Emerging/frontier', '2000-01-01', 252),
    'mxx':    ('MXX.INDX',      'IPC (Mexico)',          'Emerging/frontier', '2000-01-01', 252),
    'nsei':   ('NSEI.INDX',     'Nifty 50 (India)',      'Emerging/frontier', '2000-01-01', 252),
    'ssec':   ('SSEC.INDX',     'Shanghai (China)',      'Emerging/frontier', '2000-01-01', 252),
    'btc':    ('BTC-USD.CC',    'Bitcoin',               'Crypto', '2014-09-17', 365),
    'eth':    ('ETH-USD.CC',    'Ethereum',              'Crypto', '2016-01-01', 365),
    'eurron': ('REF:EUR',       'EUR/RON (reference rate)', 'FX', '2005-07-01', 252),
}

LABELS = {k: v[1] for k, v in MARKETS.items()}
GROUPS = {k: v[2] for k, v in MARKETS.items()}
PERIODS = {k: v[4] for k, v in MARKETS.items()}

_CACHE = {}


def read_market(symbol):
    """Daily price table of one series (local copy of the course data, otherwise the GitHub repository)."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname)
    src = path if os.path.exists(path) else REPO_RAW + fname
    try:
        return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()
    except Exception as exc:
        raise RuntimeError(f'Could not load {fname} from {src}; check the course data folder or internet connection.') from exc


def read_reference_rate(currency='EUR', start='2005-07-01', end=END):
    """Official RON reference rate (BNR annual XML archives)."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(int(start[:4]), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        try:
            xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                         timeout=60).read().decode()
        except Exception as exc:
            raise RuntimeError(f'Could not load BNR reference rates for {y}: {url}. Internet access is required.') from exc
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}">([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_close(name, start=None, end=END):
    """Daily closing price, cleaned with the chapter conventions."""
    symbol, _, group, start0, _ = MARKETS[name]
    start = start or start0
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start, end=end).rename(name)
    s = read_market(symbol)['close'].loc[start:end]
    s = s[s > 0].dropna()
    if group != 'Crypto':
        s = s[s.index.dayofweek < 5]          # no weekend quotes
        s = s[s.diff() != 0]                  # no holidays filled with the previous price
    return s.rename(name)


def complete_months(x):
    """Drop the incomplete final month (last observation before the last business day of the month):
    the monthly analyses and the turn-of-the-month effect use complete months only."""
    last = x.index[-1]
    if last + pd.offsets.BMonthEnd(0) != last:
        x = x[x.index < last.to_period('M').start_time]
    return x


def log_returns(name, start=None, end=END):
    """Log returns on the series' own calendar."""
    return np.log(load_close(name, start, end)).diff().dropna().rename(name)


rets = {k: log_returns(k) for k in MARKETS if k != 'eurron'}
ORDER = list(rets)
for k in ORDER:
    print(f'{LABELS[k]:26s}', rets[k].index[0].date(), '->', rets[k].index[-1].date(), len(rets[k]), 'obs')

S&P 500 (US)               2000-01-04 -> 2026-09-18 6714 obs
Euro Stoxx 50              2000-01-04 -> 2026-09-18 6836 obs
Nikkei 225 (Japan)         2000-01-05 -> 2026-09-18 6545 obs
Hang Seng (HK)             2000-01-04 -> 2026-09-18 6583 obs
BET (Romania)              2000-01-06 -> 2026-09-18 6670 obs
WIG20 (Poland)             2000-01-04 -> 2026-09-18 6685 obs
BUX (Hungary)              2000-01-05 -> 2026-09-18 6666 obs
PX (Czechia)               2000-01-06 -> 2026-09-18 6682 obs
BIST 100 (Turkey)          2000-01-05 -> 2026-09-18 6692 obs
Bovespa (Brazil)           2000-01-04 -> 2026-09-18 6622 obs
IPC (Mexico)               2000-01-04 -> 2026-09-18 6689 obs
Nifty 50 (India)           2000-01-04 -> 2026-09-18 6606 obs
Shanghai (China)           2000-01-05 -> 2026-09-18 6474 obs
Bitcoin                    2014-09-18 -> 2026-09-18 4384 obs
Ethereum                   2016-01-02 -> 2026-09-18 3913 obs


## Teste de eficiență

- `variance_ratio`: variance ratio VR(q) Lo–MacKinlay cu $z$ homoskedastic și $z^*$ robust.
- `chow_denning`, `runs_test`, `robust_ljung_box`, `rs_hurst` (cu Anis–Lloyd), `lo_modified_rs`, `dfa_hurst`, `rolling_stat`.

Incrementele random walk-ului sînt randamentele: RW1 cere independență și distribuție identică; RW2 păstrează independența, dar permite distribuții diferite; RW3 cere covarianță serială zero și permite dependență în momentele superioare. Statistica homoskedastică z folosește ipoteze de tip RW1; z* robust permite heteroskedasticitate sub RW3 și condițiile de regularitate necesare inferenței asimptotice. O respingere privește aceste restricții asupra randamentelor, nu toate formele eficienței pieței.

VR înseamnă variance ratio; i.i.d. înseamnă independent și identic distribuit. Erorile standard (standard errors, SE) descriu incertitudinea estimatorului. OLS înseamnă ordinary least squares; HAC desemnează erori standard heteroskedasticity- and autocorrelation-consistent. DFA este detrended fluctuation analysis, iar R/S este rescaled-range analysis. AR(1) este modelul autoregresiv de ordinul 1; în event study, AR cu indice temporal înseamnă abnormal return, iar suma lor este cumulative abnormal return (CAR). Chow–Denning (CD) combină orizonturile prin distribuția Studentized Maximum Modulus (SMM).

In [ ]:
def variance_ratio(r, q):
    """Overlapping VR(q) with the Lo-MacKinlay (1988) bias corrections.

    Returns (VR, homoskedastic z, heteroskedasticity-robust z*).
    """
    x = np.asarray(r, dtype=float)
    T = len(x)
    mu = x.mean()
    e = x - mu
    var_a = (e ** 2).sum() / (T - 1)
    agg = np.convolve(x, np.ones(q), mode='valid') - q * mu        # overlapping q-period sums
    m = q * (T - q + 1) * (1 - q / T)
    var_c = (agg ** 2).sum() / m
    vr = var_c / var_a
    z_homo = (vr - 1) / np.sqrt(2 * (2 * q - 1) * (q - 1) / (3 * q * T))
    den = (e ** 2).sum() ** 2
    theta = 0.0
    for j in range(1, q):
        delta = T * (e[j:] ** 2 * e[:-j] ** 2).sum() / den
        theta += (2 * (q - j) / q) ** 2 * delta
    z_star = np.sqrt(T) * (vr - 1) / np.sqrt(theta)
    return vr, z_homo, z_star


def chow_denning(r, qs=(2, 5, 10, 20)):
    """Chow-Denning multiple test: max |z*(q)|; p-value from the studentized maximum modulus (infinite df)."""
    zs = [variance_ratio(r, q)[2] for q in qs]
    zmax = np.max(np.abs(zs))
    p = 1 - (2 * stats.norm.cdf(zmax) - 1) ** len(qs)
    return zmax, p, zs


def runs_test(r):
    """Number of sign runs (zeros dropped), with its mean and variance under independence."""
    s = np.sign(np.asarray(r, dtype=float))
    s = s[s != 0]
    n1, n2 = (s > 0).sum(), (s < 0).sum()
    n = n1 + n2
    runs = 1 + (s[1:] != s[:-1]).sum()
    mean = 2 * n1 * n2 / n + 1
    var = 2 * n1 * n2 * (2 * n1 * n2 - n) / (n ** 2 * (n - 1))
    z = (runs - mean) / np.sqrt(var)
    return runs, mean, z, 2 * (1 - stats.norm.cdf(abs(z)))


def robust_ljung_box(r, m=10):
    """Ljung-Box Q(m) and Q~(m) with the robust variance tau_k = sum e_t^2 e_{t-k}^2 / (sum e_t^2)^2."""
    e = np.asarray(r, dtype=float) - np.mean(r)
    T = len(e)
    s2 = (e ** 2).sum()
    q_lb, q_rob = 0.0, 0.0
    for k in range(1, m + 1):
        rho = (e[k:] * e[:-k]).sum() / s2
        tau = (e[k:] ** 2 * e[:-k] ** 2).sum() / s2 ** 2
        q_lb += T * (T + 2) * rho ** 2 / (T - k)
        q_rob += rho ** 2 / tau
    return q_lb, 1 - stats.chi2.cdf(q_lb, m), q_rob, 1 - stats.chi2.cdf(q_rob, m)


def _rs(x):
    y = np.cumsum(x - x.mean())
    s = x.std(ddof=0)
    return (y.max() - y.min()) / s if s > 0 else np.nan


def expected_rs(n):
    """E[R/S] for i.i.d. noise (Anis & Lloyd, 1976)."""
    if n <= 340:
        f = np.exp(gammaln((n - 1) / 2) - gammaln(n / 2)) / np.sqrt(np.pi)
    else:
        f = 1 / np.sqrt(n * np.pi / 2)
    i = np.arange(1, n)
    return f * np.sum(np.sqrt((n - i) / i))


def _integer_log_scales(first, last, count):
    """Floor log-spaced scales; snap floating-point roundoff at exact integers."""
    raw = np.logspace(np.log10(first), np.log10(last), count)
    nearest = np.rint(raw)
    tolerance = 8 * np.spacing(np.maximum(1.0, np.abs(raw)))
    stable = np.where(np.abs(raw - nearest) <= tolerance, nearest, raw)
    return np.unique(np.floor(stable).astype(int))


def rs_hurst(r, min_n=10, n_sizes=20):
    """H from the slope of log(R/S) on log(n) (Hurst, 1951) and the Anis-Lloyd corrected H (slope of log(R/S) - log E[R/S] on log n, plus 0.5)."""
    x = np.asarray(r, dtype=float)
    T = len(x)
    sizes = _integer_log_scales(min_n, T // 4, n_sizes)
    rs, ers = [], []
    for n in sizes:
        k = T // n
        vals = [_rs(x[i * n:(i + 1) * n]) for i in range(k)]
        rs.append(np.nanmean(vals))
        ers.append(expected_rs(n))
    ls, lrs, lers = np.log(sizes), np.log(rs), np.log(ers)
    h = np.polyfit(ls, lrs, 1)[0]
    h_al = 0.5 + np.polyfit(ls, lrs - lers, 1)[0]
    return h, h_al, pd.DataFrame({'n': sizes, 'rs': rs, 'expected_rs': ers})


def lo_modified_rs(r, q=None):
    """Lo's (1991) statistic V = Q_T / sqrt(T); under H0 (no long memory) V lies in [0.809, 1.862] with 95% probability."""
    x = np.asarray(r, dtype=float)
    T = len(x)
    e = x - x.mean()
    if q is None:                                   # Andrews (1991) bandwidth for an AR(1)
        rho = np.corrcoef(e[1:], e[:-1])[0, 1]
        q = int(np.floor((3 * T / 2) ** (1 / 3) * abs(2 * rho / (1 - rho ** 2)) ** (2 / 3)))
    s2 = (e ** 2).mean()
    for j in range(1, q + 1):
        w = 1 - j / (q + 1)
        s2 += 2 * w * (e[j:] * e[:-j]).sum() / T
    y = np.cumsum(e)
    Q = (y.max() - y.min()) / np.sqrt(s2)
    return Q / np.sqrt(T), q


def dfa_hurst(r, min_box=10, n_sizes=18, max_frac=0.25):
    """DFA-1 exponent: slope of log F(n) on log n for the cumulative profile of demeaned returns."""
    x = np.asarray(r, dtype=float)
    y = np.cumsum(x - x.mean())
    T = len(y)
    sizes = _integer_log_scales(min_box, int(T * max_frac), n_sizes)
    F = []
    for n in sizes:
        k = T // n
        seg = y[:k * n].reshape(k, n)
        t = np.arange(n)
        A = np.vstack([t, np.ones(n)]).T
        coef, *_ = np.linalg.lstsq(A, seg.T, rcond=None)
        resid = seg.T - A @ coef
        F.append(np.sqrt((resid ** 2).mean()))
    return np.polyfit(np.log(sizes), np.log(F), 1)[0], pd.DataFrame({'n': sizes, 'F': F})


def rolling_stat(r, func, window=500, step=21):
    """Apply func on rolling windows of `window` observations, every `step` observations."""
    x = r.values
    idx, out = [], []
    for end in range(window, len(x) + 1, step):
        idx.append(r.index[end - 1])
        out.append(func(x[end - window:end]))
    return pd.Series(out, index=idx)

## Funcțiile seminarului

- Funcțiile folosite de exercițiile rezolvate.

In [ ]:
HERE = TABLE_DIR   # folderul tabelelor scrise de funcții
DAYS = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri']
A_RETURNS = np.array([1.0, -0.5, 0.8, 0.3, -1.2, 0.6, 0.9, -0.4])   # %
A_SIGNS = '++-+++--+-++---+-+++'
A6_SIGNS = '+++++-----+++++-----'


def a_vr_by_hand(r=A_RETURNS, q=2):
    """VR(q) without bias corrections: var(overlapping q-day sums) / (q * daily var)."""
    mu = r.mean()
    var1 = ((r - mu) ** 2).mean()
    sums = np.convolve(r, np.ones(q), mode='valid')
    varq = ((sums - q * mu) ** 2).mean()
    e = r - mu
    rho = [(e[j:] * e[:-j]).sum() / (e ** 2).sum() for j in range(1, q)]
    approx = 1 + 2 * sum((1 - j / q) * rho[j - 1] for j in range(1, q))
    return dict(mean=mu, var1=var1, sums=sums, varq=varq, VR=varq / (q * var1), rho=rho, approx=approx)


def a_runs(signs=A_SIGNS):
    n1, n2 = signs.count('+'), signs.count('-')
    n = n1 + n2
    runs = 1 + sum(signs[i] != signs[i - 1] for i in range(1, n))
    mean = 2 * n1 * n2 / n + 1
    var = 2 * n1 * n2 * (2 * n1 * n2 - n) / (n ** 2 * (n - 1))
    z = (runs - mean) / np.sqrt(var)
    return dict(n1=n1, n2=n2, runs=runs, mean=mean, var=var, z=z, p=2 * (1 - stats.norm.cdf(abs(z))))


def h_from_vr(vr, q):
    """For self-similar increments, VR(q) = q^(2H-1), so H = 0.5 + ln VR(q) / (2 ln q)."""
    return 0.5 + np.log(vr) / (2 * np.log(q))


def b_vr_table(names):
    rows = []
    for k in names:
        r = log_returns(k) if k != 'bettr' else np.log(read_market('BETTR.INDX')['close']).diff().dropna()
        v2, v5, v10, v20 = (variance_ratio(r, q) for q in (2, 5, 10, 20))
        cd = chow_denning(r)
        rows.append(dict(market=LABELS.get(k, 'BET-TR (Romania)'), N=len(r),
                         VR2=v2[0], z2=v2[1], zstar2=v2[2], VR5=v5[0], z5=v5[1], zstar5=v5[2],
                         VR10=v10[0], z10=v10[1], zstar10=v10[2],
                         VR20=v20[0], z20=v20[1], zstar20=v20[2], CD=cd[0], CD_p=cd[1]))
    return pd.DataFrame(rows).set_index('market')


def b_rolling_dfa(k='sp500', window=500, step=21, n_boot=199, seed=SEED):
    """Rolling DFA; wild-bootstrap null band (Rademacher signs) computed for EACH window:
    keeps the window's volatility (allowed under RW3) and destroys the autocorrelation of returns.
    For comparison: the i.i.d. Student-t4 band (the same for all windows)."""
    rng = np.random.default_rng(seed)
    null = [dfa_hurst(rng.standard_t(4, window))[0] for _ in range(300)]
    lo_t, hi_t = np.percentile(null, [2.5, 97.5])
    r = log_returns(k)
    x = r.values
    rows = []
    for end in range(window, len(x) + 1, step):
        e = x[end - window:end] - x[end - window:end].mean()
        g = np.random.default_rng(seed + end)
        sims = [dfa_hurst(e * g.choice([-1.0, 1.0], window))[0] for _ in range(n_boot)]
        lo, hi = np.percentile(sims, [2.5, 97.5])
        rows.append((r.index[end - 1], dfa_hurst(e)[0], lo, hi))
    d = pd.DataFrame(rows, columns=['date', 'dfa', 'lo', 'hi']).set_index('date')
    d.to_csv(os.path.join(HERE, 'ch2_sem_rolling_dfa.csv'))
    h = d['dfa']
    return dict(lo_t=lo_t, hi_t=hi_t, share_t=float(((h < lo_t) | (h > hi_t)).mean()),
                share_out=float(((h < d['lo']) | (h > d['hi'])).mean()), n=len(d),
                band_lo_mean=float(d['lo'].mean()), band_hi_mean=float(d['hi'].mean()),
                width_min=float((d['hi'] - d['lo']).min()), width_max=float((d['hi'] - d['lo']).max()),
                argwide=(d['hi'] - d['lo']).idxmax().date(), min=h.min(), max=h.max(),
                argmin=h.idxmin().date(), argmax=h.idxmax().date(), mean=h.mean())


def b_tom_ols_hac(k='sp500'):
    """Turn-of-the-month effect: the same regression with classical OLS errors and with HAC errors (Newey-West, 5 lags)."""
    r = complete_months(log_returns(k)) * 1e4
    d = pd.DataFrame({'r': r})
    ym = d.index.to_period('M')
    rank = d.groupby(ym).cumcount()
    rank_end = d.groupby(ym).cumcount(ascending=False)
    d['tom'] = ((rank < 3) | (rank_end == 0)).astype(float)
    X = sm.add_constant(d['tom'])
    ols = sm.OLS(d['r'], X).fit()
    hac = sm.OLS(d['r'], X).fit(cov_type='HAC', cov_kwds={'maxlags': 5})
    return dict(N=len(d), share_tom=float(d['tom'].mean()), const=float(ols.params['const']),
                beta=float(ols.params['tom']), se_ols=float(ols.bse['tom']), t_ols=float(ols.tvalues['tom']),
                se_hac=float(hac.bse['tom']), t_hac=float(hac.tvalues['tom']), p_hac=float(hac.pvalues['tom']),
                mean_tom=float(d.loc[d['tom'] == 1, 'r'].mean()), mean_other=float(d.loc[d['tom'] == 0, 'r'].mean()))

## Partea A — pe hîrtie

### A1. Raportul dispersiilor, pas cu pas [Rezolvat]

- Randamente (%): 1,0; −0,5; 0,8; 0,3; −1,2; 0,6; 0,9; −0,4.
- Calculați $\hat\mu$, $\hat\sigma^2_1$, sumele pe 2 zile suprapuse, $\hat\sigma^2_2$ și $\widehat{\text{VR}}(2)$; comparați cu $1 + \hat\rho_1$.

In [ ]:
# Rezolvare
a1 = a_vr_by_hand()
{k: np.round(v, 4) for k, v in a1.items()}

{'mean': np.float64(0.1875),
 'var1': np.float64(0.5586),
 'sums': array([ 0.5,  0.3,  1.1, -0.9, -0.6,  1.5,  0.5]),
 'varq': np.float64(0.6292),
 'VR': np.float64(0.5632),
 'rho': array([-0.3947]),
 'approx': np.float64(0.6053)}

### A4. Deduceți raportul dispersiilor și distribuția lui nulă [Propus]

- Deduceți $\text{VR}(q) = 1 + 2\sum_{k<q}(1 - k/q)\rho_k$ și, prin metoda delta, $\sqrt T(\widehat{\text{VR}}(q) - 1) \to N(0, 2(2q-1)(q-1)/(3q))$ sub RW1; verificați $z(2)$ pentru S&P 500. Model: A1.

In [ ]:
# Codul vostru aici


### A5. Testul secvențelor, pas cu pas [Rezolvat]

- Semnele: `++-+++--+-++---+-+++`; numărați $n_1$, $n_2$, $R$; calculați $z$ și p-value-ul.

In [ ]:
# Rezolvare
a_runs()

{'n1': 12,
 'n2': 8,
 'runs': 11,
 'mean': 10.6,
 'var': 4.3452631578947365,
 'z': np.float64(0.1918898262711097),
 'p': np.float64(0.8478285068142399)}

### A6. Deplasarea Stambaugh [Propus]

- $E[\hat\beta - \beta] = \phi E[\hat\rho - \rho] \approx -\phi(1 + 3\rho)/T$, $\phi = \sigma_{uv}/\sigma_v^2$; date anuale $\rho = 0{,}98$, $T = 60$, corr$(u, v) = -0{,}9$, $\sigma_u = 0{,}17$, $\sigma_v = 0{,}14$.
- Calculați $\phi$, deplasarea lui $\hat\beta$, $\text{SE}(\hat\beta) \approx \sigma_u/(\sigma_x\sqrt T)$ cu $\sigma_x = \sigma_v/\sqrt{1 - \rho^2}$ (aproximare asimptotică, predictor AR(1) staționar) și deplasarea în unități ale acestei SE asimptotice; apoi simulați 10 000 de selecții și comparați cu abaterea standard simulată a lui $\hat\beta$ (SE asimptotică o subestimează cînd $\rho$ este aproape de 1 și $T$ este mic); interpretați semnul deplasării. Model: exemplul Stambaugh lucrat din [Capitolul 2](https://danpele.github.io/MFM/RO/Cursuri/capitol2_eficienta_pietei.pdf#ch2adv-regression).

Pentru extensia Monte Carlo puneți alpha = c = beta = 0 în y_t = alpha + beta x_(t-1) + u_t și x_t = c + rho x_(t-1) + v_t. Generați inovații bivariate Normale independente în timp, cu abaterile standard și corelația indicate. Generați x_0 din distribuția Normală staționară, cu media zero și varianța sigma_v^2/(1-rho^2), independent de inovațiile viitoare. Folosiți seed 42 și exact T perechi predictor–randament în fiecare replicare.

In [ ]:
# Codul vostru aici


### A2. Exponentul Hurst dintr-un raport de dispersii [Rezolvat]

- $H(q) = 0{,}5 + \ln\text{VR}(q)/(2\ln q)$ implicat, exact doar pentru creșteri (incremente) autosimilare (zgomot gaussian fracționar); $\widehat{\text{VR}}(20)$ pentru BET. Un singur orizont nu dovedește memoria lungă.

In [ ]:
# Rezolvare
vr_bet = variance_ratio(rets['bet'], 20)[0]
print(round(vr_bet, 4), round(h_from_vr(vr_bet, 20), 4))

1.377 0.5534


### A3. Exponentul Hurst pentru S&P 500 [Propus]

- $H(20)$ implicat pentru S&P 500; apoi VR(20) pentru un AR(1) cu $\rho_1 = -0{,}1$ și limita lui $H(q)$ cînd $q \to \infty$ pentru memorie scurtă cu $f(0) > 0$. Model: A2.

In [ ]:
# Codul vostru aici


## Partea B — calcul și inferență

### B1. Raportul dispersiilor homoskedastic și robust [Rezolvat]

- S&P 500 și BET: VR(q), $z$ și $z^*$ pentru $q = 2, 5, 10, 20$; Chow–Denning.
- Interpretare:
  - de ce $|z| > |z^*|$?
  - ați tranzacționa pe inversarea S&P 500?

In [ ]:
# Rezolvare
b_vr_table(['sp500', 'bet']).round(3).T

market,S&P 500 (US),BET (Romania)
N,6714.000,6670.000
VR2,0.901,1.108
z2,-8.120,8.827
zstar2,-3.644,3.836
VR5,0.830,1.173
z5,-6.375,6.445
zstar5,-2.772,2.980
VR10,0.767,1.225
z10,-5.654,5.444
zstar10,-2.497,2.732


### B7. BET-TR și piețele emergente [Propus]

- Repetați B1 pentru BET-TR, WIG20, BUX, PX, BIST 100, Bovespa, IPC, Nifty 50, Shanghai. Model: B1.
- Interpretare: cîte resping cu $z(2)$, cîte cu Chow–Denning?

Familia Holm cuprinde BET-TR și cei opt indici emergenți enumerați. BET-TR începe cu primul close observat, 23 septembrie 2014; comparați BET și BET-TR pe intervalul calendaristic BET-TR.

Funcția generală b_vr_table, inclusiv încărcarea BET-TR, este furnizată. Aplicați Holm la 5% celor nouă p-value-uri ale testului joint Chow–Denning, care combină orizonturile. Pentru BET-TR, reperul folosește direct diferențele logaritmice ale close-ului; spre deosebire de BET, nu elimină zilele cu close neschimbat. Comparația restrînge intervalul calendaristic BET la perioada BET-TR, fără inner join al observațiilor. Precizați aceste convenții și discutați sensibilitatea la filtre comparabile.

In [ ]:
# Codul vostru aici


### B3. Turn-of-the-month (TOM) cu erori HAC [Rezolvat]

- $r_t = a + b\,\text{TOM}_t + u_t$ cu erori OLS și HAC (5 laguri), S&P 500 și BET.
- Interpretare: de ce diferă erorile OLS și HAC?

In [ ]:
# Rezolvare
pd.DataFrame({k: b_tom_ols_hac(k) for k in ['sp500', 'bet']}).round(4)

,sp500,bet
N,6701.0000,6656.0000
share_tom,0.1910,0.1923
const,1.8650,3.1196
beta,3.2385,17.3241
se_ols,3.7723,4.3450
t_ols,0.8585,3.9871
se_hac,3.4935,4.7813
t_hac,0.9270,3.6233
p_hac,0.3539,0.0003
mean_tom,5.1035,20.4437


### B4. Cincisprezece teste de calendar și testarea multiplă [Propus]

- Cinci piețe × trei efecte; decizii brute, Bonferroni și Holm. Model: B3.
- Extensie (StepM, vezi Anexa Cursului 2): StepM Romano–Wolf cu bootstrap pe blocuri de 3 luni.

Piețe: S&P 500, BET, Euro Stoxx 50, WIG20 și BUX, folosind luni complete. Cerința principală este aplicarea corecțiilor celor 15 p-value-uri din [Seminarul 2, B4](https://danpele.github.io/MFM/RO/Seminarii/seminar2_eficienta_pietei_ro.pdf#ex-B4); recalcularea testelor este extensia de replicare. Testați egalitatea mediilor pe zile ale săptămînii cu un test Wald HAC (5 laguri), efectul ianuarie pe randamente simple lunare (HAC, 3 laguri) și turn-of-the-month ca în B3. Aplicați Bonferroni și Holm la 5% acestei familii de 15 p-value-uri. Pentru replicare, implementați testele, comparați cu reperul public din secțiunea 13 a cursului și explicați diferențele; simpla preluare a rezultatelor de referință nu îndeplinește cerința.

In [ ]:
# Codul vostru aici


### B2. Este inversarea S&P 500 un efect al crizelor? [Propus]

- $\hat\rho_1$, $t$ i.i.d. și robust, $z^*(5)$, cu și fără 1 septembrie 2008–30 iunie 2009 (inclusiv) și 15 feb.–30 iun. 2020, calculate în segmentele contigue rămase. Model: B1.

Păstrați indicele temporal original cînd eliminați crizele. Centrați cu media tuturor randamentelor păstrate și folosiți suma lor de pătrate și numărul lor. Nu formați perechi decalate sau sume pe q zile peste un interval eliminat. În corecția m folosiți numărul sumelor valide pe q zile; numărul randamentelor păstrate este distinct de numărul perechilor valide.

In [ ]:
# Codul vostru aici


### B8. Regresie predictivă și un test out-of-sample [Propus]

- Randamentul lunar în exces al S&P 500 pe raportul log dividend–preț decalat: $t$ Newey–West, corecția Stambaugh, $R^2_{OS}$ recursiv din 1970 și testul Clark–West. Model: [B3](https://danpele.github.io/MFM/RO/Seminarii/seminar2_eficienta_pietei_ro.pdf#ex-B3); exemplul Stambaugh lucrat din [Capitolul 2](https://danpele.github.io/MFM/RO/Cursuri/capitol2_eficienta_pietei.pdf#ch2adv-regression).

P este prețul mediu lunar, nu close-ul de la sfîrșitul lunii. Prezentați codul, comparați reperul din curs și interpretați diferențele.

Date: prețul mediu lunar P și dividendele D pe ultimele 12 luni din Shiller. Extindeți P prin mediile lunare ale close-urilor S&P 500, iar D prin sumele mobile ale dividendelor lunare SPY pe 12 luni. Scalați separat P și D cu media raportului Shiller/extensie în ultimele 12 luni comune. Folosiți rf = Federal Reserve Economic Data (FRED) TB3MS /100 și excess log return log((P_t + D_t/12)/P_(t-1)) − log(1 + rf_(t-1)/12). Eșantionul efectiv după laguri: februarie 1934–august 2026 (1111 luni); Newey–West cu 12 laguri; out-of-sample (OOS) cu fereastră în expansiune din ianuarie 1970. Implementați, comparați reperul public din curs și interpretați diferențele.

In [ ]:
# Codul vostru aici


### B9. Taxa bancară, TLV și BRD împreună [Propus]

- 19 decembrie 2018: $AR_0$ cu dispersia erorii de predicție, $t$ sub independență față de $t$ pe portofoliu, corelația reziduurilor, $CAR[0, +5]$. Model: [Noțiunile de event study](https://danpele.github.io/MFM/RO/Seminarii/seminar2_eficienta_pietei_ro.pdf#sem2p21); exemplul taxei bancare din [Capitolul 2](https://danpele.github.io/MFM/RO/Cursuri/capitol2_eficienta_pietei.pdf#ch2adv-event-inference); [B3](https://danpele.github.io/MFM/RO/Seminarii/seminar2_eficienta_pietei_ro.pdf#ex-B3) pentru HAC. Opțional: [exercițiul B7 din Seminarul 0](https://danpele.github.io/MFM/RO/Seminarii/seminar0_piete_financiare_ro.pdf#ex-B7).

Folosiți adjusted close pentru TLV.RO și BRD.RO, market model cu BET, fereastra de estimare [−250,−11] zile de tranzacționare față de 19 decembrie 2018 și un portofoliu cu ponderi egale în cele două bănci. Fereastra evenimentului [0,+5] conține L = 6 zile de tranzacționare, iar [+1,+5] conține L = 5 zile de tranzacționare. Este un exercițiu de replicare: implementați pașii, comparați cu reperul public din secțiunea 13 a cursului și interpretați efectul dependenței dintre bănci.

In [ ]:
# Codul vostru aici


## După curs — B5, B6

- Seminarul 2 are loc înaintea Cursului 2: aceste cerințe folosesc instrumente introduse la curs (DFA și null band-uri bootstrap pe ferestre mobile); lucrați-le după curs, înaintea seminarului următor.

### B5. DFA pe ferestre mobile pentru S&P 500 [Rezolvat]

- Ferestre de 500 de zile, pas 21; null band prin schimbări de semn (wild bootstrap) în fiecare fereastră (199 de serii), comparat cu un null band fix i.i.d. Student-$t_4$.
- Interpretare: proporția în afara null band-ului depășește 5%?

In [ ]:
# Rezolvare
b_rolling_dfa()

{'lo_t': np.float64(0.3801268053346879),
 'hi_t': np.float64(0.6146970093088779),
 'share_t': 0.06418918918918919,
 'share_out': 0.02027027027027027,
 'n': 296,
 'band_lo_mean': 0.35933945985429144,
 'band_hi_mean': 0.6437410400394504,
 'width_min': 0.20953693868719042,
 'width_max': 0.4991034153113405,
 'argwide': datetime.date(2022, 2, 11),
 'min': 0.34046995094831317,
 'max': 0.5953478622648416,
 'argmin': datetime.date(2008, 12, 5),
 'argmax': datetime.date(2022, 1, 12),
 'mean': np.float64(0.45023882385526587)}

### B6. Eficiența adaptivă a Bitcoin [Propus]

- Bitcoin din 1 ian. 2011 pînă la 18 sep. 2026, ferestre de 365 de zile, pas 30; $z^*(5)$ cu null band prin schimbări de semn (999 de replicări) pentru fiecare fereastră. Model: B5, B1.
- Interpretare: cîte respingeri ar produce hazardul?

Pentru comparația perioadelor, grupați ferestrele după data finală: 2012–2016 față de 2017–2026. Raportați numărul ferestrelor și numărul respingerilor din fiecare grup. Ferestrele suprapuse sînt dependente; ponderile respingerilor sînt descriptive, nu observații binomiale independente.

Folosiți log_returns("btc", start="2011-01-01"); fișierul Bitcoin începe la 13 iulie 2010. Pentru acest exercițiu, data implicită din 2014 este înlocuită.

In [ ]:
# Codul vostru aici


## Partea C — întrebări deschise și critica unui răspuns AI

- C1 (critica unui răspuns AI) folosește codul din B1 de mai sus.

### C2. A devenit Bursa de Valori București (BVB) mai eficientă după upgrade-ul Financial Times Stock Exchange (FTSE) din 2020? [Propus]

- Cinci ani înainte și după 21 septembrie 2020: $\hat\rho_1$, $z^*(5)$ și un interval bootstrap pe blocuri pentru schimbare, pentru BET și piețele de comparație WIG20 și BUX.

C1: folosiți răspunsul AI din [Seminarul 2, C1](https://danpele.github.io/MFM/RO/Seminarii/seminar2_eficienta_pietei_ro.pdf#ex-C1). Clasificați fiecare afirmație ca susținută, greșită, dependentă de convenție sau necalculată de codul prezentat și justificați verdictul. Definirea unei funcții fără apelarea ei nu calculează un rezultat.

C2: moving-block bootstrap cu blocuri de 20 de zile de tranzacționare, 999 de replicări și seed 42; reeșantionați independent grupele înainte/după și definiți schimbarea ca după minus înainte.

In [ ]:
# Codul vostru aici


### C3. Big data și eficiența, in-sample și out-of-sample [Propus]

- Replicați designul Martin & Nagel (2022) (secțiunea 7 a lucrării) pe cele 49 de portofolii sectoriale din Kenneth French Data Library: OLS pe ferestre mobile de 20 de ani cu 238 de predictori din randamente decalate, apoi randamentele in-sample și out-of-sample, $r_{IS}$ și $r_{OOS}$.
- Calculați media lui $r_{OOS}$ pe ferestre de 10 ani, cu benzi descriptive ±2 SE (ca în Fig. 8 din lucrare). Quantlet-ul de curs `MFM_ch2_big_data` conține codul de referință. Model: B3, B8.

In [ ]:
# Codul pentru C3
